# Subsetting, aggregation, and the retrospective archive

This notebook needs **no network**. It documents — by running them —
the behaviours that are deliberately deferred to the pyramids `PY-G`
reader (unreleased), so there are no surprises.

NWM operational files are whole-CONUS, so **any subset needs a read**.
Until `PY-G` lands, every subset path raises a clear
`NotImplementedError` naming it.

In [ ]:
from earthlens.nwm import NWM

WHOLE = dict(lat_lim=[-90, 90], lon_lim=[-180, 180])


def show(label, fn):
    try:
        fn()
        print(f'{label}: (no error)')
    except NotImplementedError as exc:
        print(f'{label}: NotImplementedError -> {str(exc)[:80]}...')

### A bounding-box crop is `PY-G`-gated

In [ ]:
bbox = NWM(start='2026-05-26', end='2026-05-26',
           variables={'chrtout': ['streamflow']},
           lat_lim=[30, 40], lon_lim=[-100, -90])
show('bbox crop', lambda: bbox.download(progress_bar=False))

### A `sites=` reach selection is `PY-G`-gated

In [ ]:
sites = NWM(start='2026-05-26', end='2026-05-26',
            variables={'chrtout': ['streamflow']},
            sites=[101, 202], **WHOLE)
show('sites=', lambda: sites.download(progress_bar=False))

### The retrospective Zarr is `PY-G`-gated (and auto-routed by date)

In [ ]:
retro = NWM(start='2000-01-01', end='2000-01-02',
            variables={'chrtout': ['streamflow']}, **WHOLE)
print('auto-routed mode for a year-2000 window:', retro._mode)
show('retrospective', lambda: retro.download(progress_bar=False))

### `aggregate=` is rejected

`chrtout` is feature-id indexed (not griddable) and a gridded `ldasout`
reduce needs a read, so the temporal aggregator is not supported.

In [ ]:
agg = NWM(start='2026-05-26', end='2026-05-26',
          variables={'chrtout': ['streamflow']}, **WHOLE)
show('aggregate=', lambda: agg.download(aggregate=object()))